# 14u CityFlow VeRi Fusion Port

Option C only: re-extract CLIP-SENet v6 and TransReID 09v v17 features on the 14h/14e B1 CityFlow tracklet stack, build the 14t-style AQE+reranked fused similarity matrix, and add it as a fourth score-fusion stream alongside the 14e B1 primary+DINOv2 anchor.

In [ ]:
import gc
import json
import math
import os
import shutil
import subprocess
import sys
import tarfile
import time
from datetime import datetime
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np

WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
INPUT_ROOT = Path("/kaggle/input")
DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"Kaggle input exists: {INPUT_ROOT.exists()}")

## 1. Clone Repo And Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
REPO_BRANCH = "feature/pretrained-ensemble"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} branch={REPO_BRANCH} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(PROJECT)])
else:
    print("Repo already present; pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))


def pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip(
    "numpy",
    "scipy",
    "pandas",
    "faiss-cpu",
    "motmetrics",
    "omegaconf",
    "rich",
    "networkx>=3.1",
    "click",
    "loguru",
    "scikit-learn",
    "timm==1.0.11",
    "open_clip_torch==2.30.0",
    "pretrainedmodels==0.7.4",
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))

import cv2
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image, ImageOps
from tqdm.auto import tqdm

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Repo ready at {PROJECT}")
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "device:", DEVICE)

## 2. Patch Stage 4 For Precomputed 14t Similarity

In [ ]:
PIPELINE_PATH = PROJECT / "src" / "stage4_association" / "pipeline.py"
pipeline_text = PIPELINE_PATH.read_text(encoding="utf-8")

if "sec_similarity: Optional[np.ndarray] = None" not in pipeline_text:
    old_decl = '''sec_embeddings: Optional[np.ndarray] = None
    sec_weight = 0.0
'''
    new_decl = '''sec_embeddings: Optional[np.ndarray] = None
    sec_similarity: Optional[np.ndarray] = None
    sec_weight = 0.0
'''
    pipeline_text = pipeline_text.replace(old_decl, new_decl)

    old_secondary_block = '''        if sec_raw.shape[0] == n:
            sec_weight = float(sec_cfg.get("weight", 0.3))
            # L2-normalize
            sec_norms = np.linalg.norm(sec_raw, axis=1, keepdims=True)
            sec_embeddings = sec_raw / np.maximum(sec_norms, 1e-8)
            logger.info(
                f"Secondary embeddings loaded: {sec_embeddings.shape[1]}D, "
                f"weight={sec_weight:.2f} (score-level fusion)"
            )
            # Apply FIC whitening to secondary embeddings separately
            if fic_cfg.get("enabled", False):
                sec_embeddings = per_camera_whiten(
                    sec_embeddings,
                    camera_ids,
                    regularisation=float(fic_cfg.get("regularisation", 3.0)),
                    min_samples=int(fic_cfg.get("min_samples", 5)),
                )
                logger.info("Applied FIC whitening to secondary embeddings")
'''
    new_secondary_block = '''        if sec_raw.ndim == 2 and sec_raw.shape == (n, n):
            sec_weight = float(sec_cfg.get("weight", 0.3))
            sec_similarity = sec_raw.astype(np.float32, copy=False)
            if not np.isfinite(sec_similarity).all():
                raise ValueError("Secondary precomputed similarity contains NaN/Inf")
            logger.info(
                f"Secondary precomputed similarity loaded: {sec_similarity.shape}, "
                f"weight={sec_weight:.2f} (score-level fusion)"
            )
        elif sec_raw.shape[0] == n:
            sec_weight = float(sec_cfg.get("weight", 0.3))
            # L2-normalize
            sec_norms = np.linalg.norm(sec_raw, axis=1, keepdims=True)
            sec_embeddings = sec_raw / np.maximum(sec_norms, 1e-8)
            logger.info(
                f"Secondary embeddings loaded: {sec_embeddings.shape[1]}D, "
                f"weight={sec_weight:.2f} (score-level fusion)"
            )
            # Apply FIC whitening to secondary embeddings separately
            if fic_cfg.get("enabled", False):
                sec_embeddings = per_camera_whiten(
                    sec_embeddings,
                    camera_ids,
                    regularisation=float(fic_cfg.get("regularisation", 3.0)),
                    min_samples=int(fic_cfg.get("min_samples", 5)),
                )
                logger.info("Applied FIC whitening to secondary embeddings")
'''
    pipeline_text = pipeline_text.replace(old_secondary_block, new_secondary_block)

    old_condition = '''    if (sec_embeddings is not None and sec_weight > 0) or (tert_embeddings is not None and tert_weight > 0):
'''
    new_condition = '''    if (sec_similarity is not None and sec_weight > 0) or (sec_embeddings is not None and sec_weight > 0) or (tert_embeddings is not None and tert_weight > 0):
'''
    pipeline_text = pipeline_text.replace(old_condition, new_condition)

    old_sec_dot = '''            if sec_embeddings is not None and sec_weight > 0:
                sim += sec_weight * float(np.dot(sec_embeddings[i], sec_embeddings[j]))
'''
    new_sec_dot = '''            if sec_similarity is not None and sec_weight > 0:
                sim += sec_weight * float(sec_similarity[i, j])
            elif sec_embeddings is not None and sec_weight > 0:
                sim += sec_weight * float(np.dot(sec_embeddings[i], sec_embeddings[j]))
'''
    pipeline_text = pipeline_text.replace(old_sec_dot, new_sec_dot)
    PIPELINE_PATH.write_text(pipeline_text, encoding="utf-8")
    print("Patched Stage 4 to accept an NxN secondary precomputed similarity matrix")
else:
    print("Stage 4 precomputed-similarity patch already present")

## 3. CLIP-SENet Architecture From 14t

In [ ]:
"""CLIP-SENet architecture for vehicle re-identification.

Milestone M1 covers only the model port and local forward-pass smoke tests.
Training losses, camera/viewpoint embeddings, and Kaggle integration are
intentionally deferred.
"""

from __future__ import annotations

from dataclasses import dataclass

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger(__name__)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)
    logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class LoadedBackboneInfo:
    """Describes the backbone variant that was successfully loaded."""

    family: str
    model_name: str
    pretrained_tag: str | None = None


class AFEMBlock(nn.Module):
    """Adaptive Fine-grained Enhancement Module.

    This implements the paper's ambiguous Eq. (4) using the `(G + 1)`
    interpretation: `G` grouped weighted residual chunks plus one identity
    residual path. Set `residual_mode="sum_only"` to drop the identity term and
    return only the weighted grouped sum.
    """

    def __init__(
        self,
        in_dim: int = 2048,
        out_dim: int = 2048,
        num_groups: int = 32,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        if out_dim % num_groups != 0:
            raise ValueError(
                f"AFEM out_dim={out_dim} must be divisible by num_groups={num_groups}"
            )
        if residual_mode not in {"grouped_identity", "sum_only"}:
            raise ValueError(
                "residual_mode must be 'grouped_identity' or 'sum_only'"
            )

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_groups = num_groups
        self.group_dim = out_dim // num_groups
        self.residual_mode = residual_mode

        self.shared = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )
        self.group_weights = nn.Parameter(torch.randn(num_groups, self.group_dim))

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        linear = self.shared[0]
        nn.init.kaiming_normal_(linear.weight, mode="fan_out")
        bn = self.shared[1]
        nn.init.ones_(bn.weight)
        nn.init.zeros_(bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared(x)
        grouped = h.view(h.shape[0], self.num_groups, self.group_dim)
        weighted = grouped * self.group_weights.unsqueeze(0)
        enhanced = weighted.reshape(h.shape[0], self.out_dim)
        if self.residual_mode == "sum_only":
            return enhanced
        return h + enhanced


class _ResNetFeatureWrapper(nn.Module):
    """Wrap torchvision-style ResNet backbones to expose pooled 2048-d features."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return torch.flatten(x, 1)


class ResNet101IBNBranch(nn.Module):
    """Appearance branch backed by real ResNet101 IBN-a with deterministic fallbacks."""

    _IBN_MODEL = "resnet101_ibn_a"
    _FALLBACK_MODEL = "resnet101"

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.output_dim = 2048
        self.backbone: nn.Module
        self.loaded_backbone: LoadedBackboneInfo

        for loader in (
            self._load_pretrainedmodels_ibn,
            self._load_torch_hub_ibn,
            self._load_timm_ibn,
            self._load_timm_plain,
        ):
            loaded = loader(pretrained=pretrained)
            if loaded is None:
                continue
            self.backbone, self.loaded_backbone = loaded
            logger.info(
                "Appearance branch loaded via '%s' model='%s' pretrained_tag='%s'",
                self.loaded_backbone.family,
                self.loaded_backbone.model_name,
                self.loaded_backbone.pretrained_tag,
            )
            return

        raise ImportError(
            "Unable to load appearance backbone via pretrainedmodels, torch.hub, or timm"
        )

    def _load_pretrainedmodels_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import pretrainedmodels
        except ImportError:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' is unavailable; trying torch.hub"
            )
            return None

        constructor = getattr(pretrainedmodels, self._IBN_MODEL, None)
        if constructor is None:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' has no '%s' entry; trying torch.hub",
                self._IBN_MODEL,
            )
            return None

        pretrained_tag = "imagenet" if pretrained else None
        try:
            raw_model = constructor(pretrained=pretrained_tag)
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "last_linear"):
            raw_model.last_linear = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="pretrainedmodels",
            model_name=self._IBN_MODEL,
            pretrained_tag=pretrained_tag or "random_init",
        )

    def _load_torch_hub_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            raw_model = torch.hub.load(
                "XingangPan/IBN-Net",
                self._IBN_MODEL,
                pretrained=pretrained,
                trust_repo=True,
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'torch.hub' failed for '{}': {}",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "fc"):
            raw_model.fc = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="torch.hub",
            model_name=self._IBN_MODEL,
            pretrained_tag="official_pretrained" if pretrained else "random_init",
        )

    def _load_timm_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        available = set(timm.list_models())
        if self._IBN_MODEL not in available:
            logger.warning(
                "Appearance branch loader 'timm' has no '%s' entry; trying plain '%s'",
                self._IBN_MODEL,
                self._FALLBACK_MODEL,
            )
            return None

        try:
            backbone = timm.create_model(
                self._IBN_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._IBN_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def _load_timm_plain(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        try:
            backbone = timm.create_model(
                self._FALLBACK_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for plain '%s': %s",
                self._FALLBACK_MODEL,
                exc,
            )
            return None

        logger.warning(
            "Appearance branch fell back to plain '%s' because no IBN-a loader succeeded",
            self._FALLBACK_MODEL,
        )
        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._FALLBACK_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        if out.ndim != 2:
            raise RuntimeError(
                f"Appearance branch expected pooled 2D output, got shape {tuple(out.shape)}"
            )
        return out


class TinyCLIPImageBranch(nn.Module):
    """Semantic branch that loads TinyCLIP with a deterministic fallback chain."""

    _OPEN_CLIP_CHAIN = (
        {
            "model_name": "hf-hub:wkcn/TinyCLIP-ViT-45M-32-Text-21M-LAION400M",
            "pretrained_tag": None,
        },
        {
            "model_name": "TinyCLIP-ViT-40M-32-Text-19M",
            "pretrained_tag": "laion400m_e32",
        },
    )
    _TIMM_TINYCLIP_CHAIN = (
        "vit_medium_patch32_clip_224.tinyclip_laion400m",
    )
    _LAST_RESORT_OPEN_CLIP = ("ViT-B-32", "openai")

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.provider = ""
        self.model = None
        self.loaded_backbone: LoadedBackboneInfo | None = None
        last_error = self._try_load_open_clip(pretrained=pretrained)
        if self.model is None:
            last_error = self._try_load_timm_tinyclip(pretrained=pretrained) or last_error
        if self.model is None:
            last_error = self._try_load_open_clip_last_resort(pretrained=pretrained) or last_error

        if self.model is None or self.loaded_backbone is None:
            raise RuntimeError(
                "Unable to load any TinyCLIP/OpenCLIP visual backbone"
            ) from last_error

        self.image_size = self._infer_image_size(self.model)

    def _try_load_open_clip(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for candidate in self._OPEN_CLIP_CHAIN:
            model_name = candidate["model_name"]
            pretrained_tag = candidate["pretrained_tag"]
            try:
                if pretrained_tag is None:
                    model, _, _ = open_clip.create_model_and_transforms(model_name)
                else:
                    model, _, _ = open_clip.create_model_and_transforms(
                        model_name,
                        pretrained=pretrained_tag if pretrained else None,
                    )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP load failed for model='%s' pretrained='%s': %s",
                    model_name,
                    pretrained_tag or "hf-hub-default",
                    exc,
                )
                continue

            self.model = model
            self.provider = "open_clip"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag=pretrained_tag if pretrained else "random_init",
            )
            self.output_dim = self._infer_open_clip_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
                model_name,
                pretrained_tag if pretrained_tag is not None and pretrained else "hf-hub-default",
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_timm_tinyclip(self, pretrained: bool) -> Exception | None:
        try:
            import timm
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for model_name in self._TIMM_TINYCLIP_CHAIN:
            try:
                model = timm.create_model(
                    model_name,
                    pretrained=pretrained,
                    num_classes=0,
                )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP-equivalent timm load failed for model='%s': %s",
                    model_name,
                    exc,
                )
                continue

            self.model = model
            self.provider = "timm"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag="timm_pretrained" if pretrained else "random_init",
            )
            self.output_dim = self._infer_timm_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' via timm output_dim=%s",
                model_name,
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_open_clip_last_resort(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        model_name, pretrained_tag = self._LAST_RESORT_OPEN_CLIP
        try:
            model, _, _ = open_clip.create_model_and_transforms(
                model_name,
                pretrained=pretrained_tag if pretrained else None,
            )
        except Exception as exc:  # noqa: BLE001 - explicit last resort context
            logger.warning(
                "OpenCLIP last resort load failed for model='%s' pretrained='%s': %s",
                model_name,
                pretrained_tag,
                exc,
            )
            return exc

        self.model = model
        self.provider = "open_clip"
        self.loaded_backbone = LoadedBackboneInfo(
            family="semantic",
            model_name=model_name,
            pretrained_tag=pretrained_tag if pretrained else "random_init",
        )
        self.output_dim = self._infer_open_clip_output_dim(model)
        logger.info(
            "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
            model_name,
            pretrained_tag if pretrained else "random_init",
            self.output_dim,
        )
        return None

    @staticmethod
    def _infer_open_clip_output_dim(model: nn.Module) -> int:
        visual = getattr(model, "visual", None)
        output_dim = getattr(visual, "output_dim", None)
        if isinstance(output_dim, int):
            return output_dim

        visual_proj = getattr(model, "visual_projection", None)
        if isinstance(visual_proj, torch.Tensor) and visual_proj.ndim == 2:
            return int(visual_proj.shape[-1])

        visual_proj = getattr(visual, "proj", None)
        if isinstance(visual_proj, torch.Tensor):
            if visual_proj.ndim == 1:
                return int(visual_proj.shape[0])
            if visual_proj.ndim == 2:
                return int(visual_proj.shape[-1])

        raise RuntimeError("Could not infer TinyCLIP visual output dimension")

    @staticmethod
    def _infer_timm_output_dim(model: nn.Module) -> int:
        output_dim = getattr(model, "num_features", None)
        if isinstance(output_dim, int):
            return output_dim
        raise RuntimeError("Could not infer timm TinyCLIP visual output dimension")

    @staticmethod
    def _infer_image_size(model: nn.Module) -> tuple[int, int]:
        pretrained_cfg = getattr(model, "pretrained_cfg", None)
        if isinstance(pretrained_cfg, dict):
            input_size = pretrained_cfg.get("input_size")
            if isinstance(input_size, tuple) and len(input_size) == 3:
                return (int(input_size[-2]), int(input_size[-1]))

        visual = getattr(model, "visual", None)
        image_size = getattr(visual, "image_size", None)
        if isinstance(image_size, int):
            return (image_size, image_size)
        if isinstance(image_size, tuple) and len(image_size) == 2:
            return (int(image_size[0]), int(image_size[1]))
        return (224, 224)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if tuple(x.shape[-2:]) != self.image_size:
            x = F.interpolate(
                x,
                size=self.image_size,
                mode="bilinear",
                align_corners=False,
            )
        if self.provider == "open_clip":
            features = self.model.encode_image(x, normalize=False)
        else:
            features = self.model(x)
        if features.ndim != 2:
            raise RuntimeError(
                f"TinyCLIP branch expected 2D image features, got shape {tuple(features.shape)}"
            )
        return features


class CLIPSENet(nn.Module):
    """CLIP-SENet with a CNN appearance branch and a CLIP semantic branch."""

    def __init__(
        self,
        num_classes: int,
        embed_dim: int = 2048,
        afem_groups: int = 32,
        feat_dim_appearance: int = 2048,
        feat_dim_semantic: int = 512,
        dropout: float = 0.0,
        appearance_pretrained: bool = True,
        semantic_pretrained: bool = True,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        self.appearance_branch = ResNet101IBNBranch(pretrained=appearance_pretrained)
        self.semantic_branch = TinyCLIPImageBranch(pretrained=semantic_pretrained)

        detected_app_dim = self.appearance_branch.output_dim
        detected_sem_dim = self.semantic_branch.output_dim
        if feat_dim_appearance != detected_app_dim:
            logger.warning(
                "Requested feat_dim_appearance=%s but backbone reports %s. Using detected dim.",
                feat_dim_appearance,
                detected_app_dim,
            )
        if feat_dim_semantic != detected_sem_dim:
            logger.warning(
                "Requested feat_dim_semantic=%s but backbone reports %s. Using detected dim.",
                feat_dim_semantic,
                detected_sem_dim,
            )

        self.feat_dim_appearance = detected_app_dim
        self.feat_dim_semantic = detected_sem_dim
        self.fusion_fc = nn.Linear(
            self.feat_dim_appearance + self.feat_dim_semantic,
            embed_dim,
            bias=False,
        )
        self.afem = AFEMBlock(
            in_dim=embed_dim,
            out_dim=embed_dim,
            num_groups=afem_groups,
            residual_mode=residual_mode,
        )
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

        nn.init.kaiming_normal_(self.fusion_fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.loaded_resnext_model = self.appearance_branch.loaded_backbone.model_name
        self.loaded_tinyclip_model = self.semantic_branch.loaded_backbone.model_name

    def forward(self, x: torch.Tensor):
        f_app = self.appearance_branch(x)
        f_sem = self.semantic_branch(x)
        t_u = self.fusion_fc(torch.cat([f_app, f_sem], dim=1))
        t_s_prime = self.afem(t_u)
        t = t_u + t_s_prime
        t_bn = self.bnneck(t)
        t_bn_normalized = F.normalize(t_bn, p=2, dim=1)

        if self.training:
            logits = self.classifier(self.dropout(t_bn))
            return t_bn_normalized, logits

        return t_bn_normalized


def build_clip_senet(num_classes: int, **kwargs) -> CLIPSENet:
    """Build a CLIP-SENet model for M1 architecture validation."""

    return CLIPSENet(num_classes=num_classes, **kwargs)

## 4. Resolve Inputs And 14h Anchor Stack

In [ ]:
from src.core.io_utils import load_tracklets_by_camera
from src.stage2_features.crop_extractor import CropExtractor
from src.stage2_features.embeddings import l2_normalize as repo_l2_normalize

SOURCE_14H_OWNER_SLUG = "yahiaakhalafallah/14h-robust-tracklet-pooling"
SOURCE_13_OWNER_SLUG = "yahiaakhalafallah/13-clip-senet-train"
SOURCE_14H_SLUG = SOURCE_14H_OWNER_SLUG.split("/", 1)[1]
EXPECTED_CAMS = ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]
EXPECTED_TRACKLETS = 929
TRANSREID_DIM = 768
CLIPSENET_DIM = 2048


def find_input_dir(slug: str, owner_slug: str | None = None, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    if owner_slug:
        owner, _, kernel = owner_slug.partition("/")
        nested = INPUT_ROOT / "notebooks" / owner / kernel
        if nested.exists():
            return nested
    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    for path in list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or (lowered_hints and all(hint in name for hint in lowered_hints)):
            return path
    return direct


def find_14h_checkpoint() -> Path:
    source_dir = find_input_dir(SOURCE_14H_SLUG, SOURCE_14H_OWNER_SLUG, hints=("14h", "robust", "tracklet"))
    checkpoint_path = source_dir / "checkpoint.tar.gz"
    if checkpoint_path.exists():
        print(f"14h input: {source_dir}")
        return checkpoint_path
    matches = sorted(INPUT_ROOT.rglob("checkpoint.tar.gz")) if INPUT_ROOT.exists() else []
    for match in matches:
        if "14h" in str(match).lower() or "robust" in str(match).lower():
            print(f"14h checkpoint discovered: {match}")
            return match
    raise FileNotFoundError(f"14h checkpoint.tar.gz not found; visible={matches[:20]}")


checkpoint = find_14h_checkpoint()
EXTRACT_DIR = Path("/tmp/14h_checkpoint")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint), "r:gz") as archive:
    archive.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json", encoding="utf-8") as handle:
    previous_meta = json.load(handle)
SOURCE_14H_RUN_NAME = previous_meta["run_name"]
SOURCE_14H_RUN_DIR = EXTRACT_DIR / SOURCE_14H_RUN_NAME
SOURCE_STAGE1_DIR = SOURCE_14H_RUN_DIR / "stage1"
SOURCE_STAGE2_DIR = SOURCE_14H_RUN_DIR / "stage2"
for required_path in [
    SOURCE_STAGE1_DIR,
    SOURCE_STAGE2_DIR / "embeddings.npy",
    SOURCE_STAGE2_DIR / "embeddings_tertiary.npy",
    SOURCE_STAGE2_DIR / "hsv_features.npy",
    SOURCE_STAGE2_DIR / "embedding_index.json",
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
source_index_map = json.loads((SOURCE_STAGE2_DIR / "embedding_index.json").read_text(encoding="utf-8"))
if len(source_index_map) != EXPECTED_TRACKLETS:
    raise RuntimeError(f"Expected {EXPECTED_TRACKLETS} 14h rows, found {len(source_index_map)}")
print(f"Loaded 14h run: {SOURCE_14H_RUN_NAME} rows={len(source_index_map)}")

CITYFLOW_INPUT = None
for candidate in [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]:
    if candidate.exists():
        CITYFLOW_INPUT = candidate
        break
if CITYFLOW_INPUT is None:
    matches = [path for path in INPUT_ROOT.rglob("data-aicity-2023-track-2") if path.is_dir()]
    if matches:
        CITYFLOW_INPUT = matches[0]
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists():
        shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_dir = DATA_RAW / f"{scene_dir.name}_{cam_dir.name}"
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

tracklets_by_camera = load_tracklets_by_camera(SOURCE_STAGE1_DIR)
total_tracklets = sum(len(tracklets) for tracklets in tracklets_by_camera.values())
if total_tracklets != EXPECTED_TRACKLETS:
    raise RuntimeError(f"Expected {EXPECTED_TRACKLETS} tracklets, found {total_tracklets}")
video_paths = {}
for cam_dir in sorted(DATA_RAW.glob("S*_c*")):
    video_path = cam_dir / "vdo.avi"
    if video_path.exists():
        video_paths[cam_dir.name] = str(video_path)
missing_videos = sorted(set(tracklets_by_camera) - set(video_paths))
if missing_videos:
    raise FileNotFoundError(f"Missing videos for cameras: {missing_videos}")
print(json.dumps({"tracklets": total_tracklets, "cameras": {cam: len(tracklets_by_camera[cam]) for cam in sorted(tracklets_by_camera)}}, indent=2))

## 5. Load 14t Parent Models

In [ ]:
from src.stage2_features.transreid_model import build_transreid

SIE_NUM_CAMERAS = 20
CONCAT_PATCH_GEM_P = 3.0
TRANSREID_IMG_SIZE = (224, 224)
CLIPSENET_IMG_SIZE = (224, 224)
TRANSREID_BATCH_SIZE = 64
CLIPSENET_BATCH_SIZE = 32
QUALITY_TEMPERATURE = 3.0
SAMPLES_PER_TRACKLET = 48

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def find_file_under_input(filename: str, preferred_slug: str | None = None) -> Path:
    candidates = []
    for path in INPUT_ROOT.rglob(filename):
        score = 100 if preferred_slug and preferred_slug in str(path) else 0
        candidates.append((score, path))
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    candidates.sort(key=lambda row: (-row[0], str(row[1])))
    for score, path in candidates[:10]:
        print(f"candidate {filename}: score={score:3d} {path}")
    return candidates[0][1]


def discover_clip_senet_checkpoint() -> Path:
    names = ("vehicle_clip_senet_veri776.pth", "best_mAP.pth", "best.pth")
    candidates = []
    for name in names:
        for path in INPUT_ROOT.rglob(name):
            score = 0
            path_text = str(path)
            if "13-clip-senet-train" in path_text:
                score += 100
            if name == "vehicle_clip_senet_veri776.pth":
                score += 20
            if name == "best_mAP.pth":
                score += 10
            candidates.append((score, path))
    if not candidates:
        raise FileNotFoundError("Could not find CLIP-SENet v6 checkpoint from 13-clip-senet-train")
    candidates.sort(key=lambda row: (-row[0], str(row[1])))
    for score, path in candidates[:10]:
        print(f"candidate CLIP-SENet: score={score:3d} {path}")
    return candidates[0][1]


TRANSREID_WEIGHTS = find_file_under_input("vehicle_transreid_vit_base_veri776.pth", preferred_slug="mtmc-weights")
CLIPSENET_WEIGHTS = discover_clip_senet_checkpoint()


def load_transreid_model(weights_path: Path):
    model = build_transreid(
        num_classes=1,
        num_cameras=SIE_NUM_CAMERAS,
        embed_dim=TRANSREID_DIM,
        vit_model="vit_base_patch16_clip_224.openai",
        pretrained=False,
        weights_path=str(weights_path),
        img_size=TRANSREID_IMG_SIZE,
    )
    model._concat_patch = False
    model._gem_p = CONCAT_PATCH_GEM_P
    return model.to(DEVICE).eval()


def load_clip_senet_model(weights_path: Path):
    payload = torch_load(weights_path)
    if isinstance(payload, dict) and "model_state" in payload:
        state_dict = payload["model_state"]
        checkpoint_kind = "payload:model_state"
    elif isinstance(payload, dict) and "model" in payload and isinstance(payload["model"], dict):
        state_dict = payload["model"]
        checkpoint_kind = "payload:model"
    elif isinstance(payload, dict) and payload and all(hasattr(value, "shape") for value in payload.values()):
        state_dict = payload
        checkpoint_kind = "state_dict"
    else:
        raise TypeError(f"Unsupported checkpoint format at {weights_path}: {type(payload).__name__}")
    classifier_weight = state_dict.get("classifier.weight")
    inferred_num_classes = int(classifier_weight.shape[0]) if classifier_weight is not None else 575
    model = build_clip_senet(num_classes=inferred_num_classes).to(DEVICE)
    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
    if missing_keys or unexpected_keys:
        print("Missing keys:", missing_keys)
        print("Unexpected keys:", unexpected_keys)
        raise RuntimeError("CLIP-SENet checkpoint load was not strict")
    model.eval()
    return model, {"checkpoint_kind": checkpoint_kind, "num_classes": inferred_num_classes}


transreid_model = load_transreid_model(TRANSREID_WEIGHTS)
clipsenet_model, clipsenet_load_info = load_clip_senet_model(CLIPSENET_WEIGHTS)

MODEL_INFO = {
    "transreid_weights": str(TRANSREID_WEIGHTS),
    "clipsenet_weights": str(CLIPSENET_WEIGHTS),
    "clipsenet_load_info": clipsenet_load_info,
    "transreid_img_size": list(TRANSREID_IMG_SIZE),
    "clipsenet_img_size": list(CLIPSENET_IMG_SIZE),
    "tta_views": ["original", "hflip", "scale_0.95", "scale_1.05"],
}
print(json.dumps(MODEL_INFO, indent=2))

## 6. Extract Dual 4-View TTA Tracklet Features

In [ ]:
FEATURE_RUN_DIR = Path("/kaggle/working/outputs/14u_features")
FEATURE_RUN_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR = FEATURE_RUN_DIR / "stage2"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR = FEATURE_DIR / "per_camera"
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

camera_to_sie = {camera_id: index for index, camera_id in enumerate(sorted(tracklets_by_camera))}
row_by_tracklet = {}
for row_index, record in enumerate(source_index_map):
    key = (str(record["camera_id"]), str(record["track_id"]))
    if key in row_by_tracklet:
        raise RuntimeError(f"Duplicate source embedding_index key: {key}")
    row_by_tracklet[key] = row_index

transreid_transform = T.Compose([
    T.Resize(TRANSREID_IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])
clipsenet_transform = T.Compose([
    T.Resize(CLIPSENET_IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def pil_from_bgr(image_bgr: np.ndarray) -> Image.Image:
    return Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))


def scaled_canvas(image: Image.Image, scale: float) -> Image.Image:
    width, height = image.size
    new_size = (max(1, int(round(width * scale))), max(1, int(round(height * scale))))
    resized = image.resize(new_size, Image.Resampling.BICUBIC)
    if scale >= 1.0:
        left = max(0, (resized.width - width) // 2)
        top = max(0, (resized.height - height) // 2)
        return resized.crop((left, top, left + width, top + height))
    canvas = Image.new("RGB", (width, height), (0, 0, 0))
    left = (width - resized.width) // 2
    top = (height - resized.height) // 2
    canvas.paste(resized, (left, top))
    return canvas


def tta_views(image: Image.Image) -> list[Image.Image]:
    return [
        image,
        ImageOps.mirror(image),
        scaled_canvas(image, 0.95),
        scaled_canvas(image, 1.05),
    ]


def softmax_quality_pool(crop_features: np.ndarray, qualities: np.ndarray) -> np.ndarray:
    logits = qualities.astype(np.float32) * float(QUALITY_TEMPERATURE)
    logits = logits - logits.max()
    weights = np.exp(logits).astype(np.float32)
    weights = weights / max(float(weights.sum()), 1e-8)
    pooled = (crop_features * weights[:, np.newaxis]).sum(axis=0).astype(np.float32)
    return repo_l2_normalize(pooled[np.newaxis, :])[0]


@torch.no_grad()
def extract_transreid_tracklet(scored_crops, sie_index: int) -> np.ndarray | None:
    if not scored_crops:
        return None
    tensors = []
    qualities = []
    for scored_crop in scored_crops:
        image = pil_from_bgr(scored_crop.image)
        qualities.append(float(scored_crop.quality))
        for view in tta_views(image):
            tensors.append(transreid_transform(view))
    view_features = []
    transreid_model.eval()
    transreid_model._concat_patch = False
    transreid_model._gem_p = CONCAT_PATCH_GEM_P
    for start in range(0, len(tensors), TRANSREID_BATCH_SIZE):
        images = torch.stack(tensors[start:start + TRANSREID_BATCH_SIZE], dim=0).to(DEVICE, non_blocking=True)
        sie = torch.full((images.shape[0],), int(sie_index), dtype=torch.long, device=DEVICE)
        with torch.cuda.amp.autocast(enabled=DEVICE.startswith("cuda")):
            output = transreid_model(images, cam_ids=sie)
        if isinstance(output, (tuple, list)):
            output = output[-1]
        view_features.append(F.normalize(output.float(), p=2, dim=1).cpu().numpy().astype(np.float32))
    views = np.concatenate(view_features, axis=0).reshape(len(scored_crops), 4, -1)
    crop_features = repo_l2_normalize(views.mean(axis=1).astype(np.float32))
    return softmax_quality_pool(crop_features, np.asarray(qualities, dtype=np.float32))


@torch.no_grad()
def extract_clipsenet_tracklet(scored_crops) -> np.ndarray | None:
    if not scored_crops:
        return None
    tensors = []
    qualities = []
    for scored_crop in scored_crops:
        image = pil_from_bgr(scored_crop.image)
        qualities.append(float(scored_crop.quality))
        for view in tta_views(image):
            tensors.append(clipsenet_transform(view))
    view_features = []
    clipsenet_model.eval()
    for start in range(0, len(tensors), CLIPSENET_BATCH_SIZE):
        images = torch.stack(tensors[start:start + CLIPSENET_BATCH_SIZE], dim=0).to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.startswith("cuda")):
            output = clipsenet_model(images)
        if isinstance(output, (tuple, list)):
            output = output[-1]
        view_features.append(F.normalize(output.float(), p=2, dim=1).cpu().numpy().astype(np.float32))
    views = np.concatenate(view_features, axis=0).reshape(len(scored_crops), 4, -1)
    crop_features = repo_l2_normalize(views.mean(axis=1).astype(np.float32))
    return softmax_quality_pool(crop_features, np.asarray(qualities, dtype=np.float32))


def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


crop_extractor = CropExtractor(samples_per_tracklet=SAMPLES_PER_TRACKLET, min_area=32 * 32, min_quality=0.05)
transreid_matrix = np.zeros((EXPECTED_TRACKLETS, TRANSREID_DIM), dtype=np.float32)
clipsenet_matrix = np.zeros((EXPECTED_TRACKLETS, CLIPSENET_DIM), dtype=np.float32)
processed_mask = np.zeros((EXPECTED_TRACKLETS,), dtype=bool)
filled_mask = np.zeros((EXPECTED_TRACKLETS,), dtype=bool)
dropped_tracklets = []
per_camera = {}
start_time = time.time()

for camera_id in sorted(tracklets_by_camera):
    camera_start = time.time()
    tracklets = tracklets_by_camera[camera_id]
    video_path = video_paths[camera_id]
    camera_rows = []
    camera_drops = []
    camera_filled = 0
    print(f"Extracting {camera_id}: {len(tracklets)} tracklets")
    for tracklet in tqdm(tracklets, desc=camera_id):
        key = (str(camera_id), str(tracklet.track_id))
        if key not in row_by_tracklet:
            raise RuntimeError(f"Tracklet missing from 14h embedding index: {key}")
        row_index = row_by_tracklet[key]
        camera_rows.append(row_index)
        processed_mask[row_index] = True
        scored_crops = crop_extractor.extract_crops(tracklet, video_path)
        if not scored_crops:
            drop = {"index": int(row_index), "camera_id": camera_id, "track_id": int(tracklet.track_id), "reason": "no_crops_survived_filtering"}
            camera_drops.append(drop)
            dropped_tracklets.append(drop)
            continue
        trans_embedding = extract_transreid_tracklet(scored_crops, camera_to_sie[camera_id])
        clip_embedding = extract_clipsenet_tracklet(scored_crops)
        if trans_embedding is None or clip_embedding is None:
            drop = {"index": int(row_index), "camera_id": camera_id, "track_id": int(tracklet.track_id), "reason": "model_returned_no_embedding"}
            camera_drops.append(drop)
            dropped_tracklets.append(drop)
            continue
        transreid_matrix[row_index] = trans_embedding.astype(np.float32)
        clipsenet_matrix[row_index] = clip_embedding.astype(np.float32)
        filled_mask[row_index] = True
        camera_filled += 1
    camera_elapsed = time.time() - camera_start
    per_camera[camera_id] = {"tracklets": len(tracklets), "filled": int(camera_filled), "dropped": int(len(camera_drops)), "elapsed_minutes": round(camera_elapsed / 60.0, 2)}
    np.save(PARTIAL_DIR / f"{camera_id}_transreid09v.npy", transreid_matrix[np.asarray(camera_rows, dtype=np.int64)])
    np.save(PARTIAL_DIR / f"{camera_id}_clipsenet.npy", clipsenet_matrix[np.asarray(camera_rows, dtype=np.int64)])
    write_json(PARTIAL_DIR / f"{camera_id}_dropped.json", camera_drops)
    write_json(FEATURE_DIR / "partial_status.json", {"per_camera": per_camera, "filled_rows": int(filled_mask.sum()), "processed_rows": int(processed_mask.sum()), "dropped_rows": int(len(dropped_tracklets))})
    print(f"  {camera_id}: filled={camera_filled} dropped={len(camera_drops)} elapsed={camera_elapsed / 60.0:.1f} min")

if not processed_mask.all():
    missing_rows = np.flatnonzero(~processed_mask).astype(int).tolist()
    raise RuntimeError(f"Rows present in 14h index were not visited: {missing_rows[:20]}")

transreid_matrix = repo_l2_normalize(transreid_matrix).astype(np.float32)
clipsenet_matrix = repo_l2_normalize(clipsenet_matrix).astype(np.float32)
if transreid_matrix.shape != (EXPECTED_TRACKLETS, TRANSREID_DIM):
    raise RuntimeError(f"Unexpected TransReID matrix shape: {transreid_matrix.shape}")
if clipsenet_matrix.shape != (EXPECTED_TRACKLETS, CLIPSENET_DIM):
    raise RuntimeError(f"Unexpected CLIP-SENet matrix shape: {clipsenet_matrix.shape}")
if not np.isfinite(transreid_matrix).all() or not np.isfinite(clipsenet_matrix).all():
    raise RuntimeError("Feature matrices contain NaN/Inf")

np.save(FEATURE_DIR / "transreid09v_cityflow_14u.npy", transreid_matrix)
np.save(FEATURE_DIR / "clipsenet_cityflow_14u.npy", clipsenet_matrix)
write_json(FEATURE_DIR / "embedding_index.json", source_index_map)
write_json(FEATURE_DIR / "dropped_tracklets.json", sorted(dropped_tracklets, key=lambda item: item["index"]))
write_json(FEATURE_DIR / "feature_summary.json", {
    "experiment": "14u-dual-feature-extraction",
    "source_14h_run_name": SOURCE_14H_RUN_NAME,
    "feature_dir": str(FEATURE_DIR),
    "transreid_shape": list(transreid_matrix.shape),
    "clipsenet_shape": list(clipsenet_matrix.shape),
    "filled_rows": int(filled_mask.sum()),
    "dropped_rows": int(len(dropped_tracklets)),
    "dropped_tracklets": sorted(dropped_tracklets, key=lambda item: item["index"]),
    "per_camera": per_camera,
    "model_info": MODEL_INFO,
    "elapsed_minutes": round((time.time() - start_time) / 60.0, 2),
})

del transreid_model, clipsenet_model
gc.collect()
if DEVICE.startswith("cuda"):
    torch.cuda.empty_cache()
print("Saved dual feature matrices:", FEATURE_DIR)

## 7. Build 14t-Style Fused Similarity Matrix

In [ ]:
def l2_normalize(features: np.ndarray) -> np.ndarray:
    features = features.astype(np.float32, copy=False)
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    return features / np.maximum(norms, 1e-12)


def average_query_expansion(features: np.ndarray, k: int, iterations: int = 1) -> np.ndarray:
    current = l2_normalize(features.copy())
    if k <= 1:
        return current
    for _ in range(iterations):
        sim = current @ current.T
        topk = min(k, sim.shape[1])
        kth = max(topk - 1, 0)
        topk_idx = np.argpartition(-sim, kth=kth, axis=1)[:, :topk]
        expanded = np.zeros_like(current, dtype=np.float32)
        for index in range(current.shape[0]):
            expanded[index] = current[topk_idx[index]].mean(axis=0)
        current = l2_normalize(expanded)
    return current


@torch.no_grad()
def build_rerank_state_from_similarity(similarity: np.ndarray, max_k1: int):
    sim_tensor = torch.as_tensor(similarity, dtype=torch.float32, device=DEVICE)
    original_dist = (2.0 - 2.0 * sim_tensor).clamp_min_(0).cpu().numpy().astype(np.float32)
    initial_rank = torch.topk(
        sim_tensor,
        k=min(max_k1 + 1, sim_tensor.shape[1]),
        dim=1,
        largest=True,
        sorted=True,
    ).indices.cpu().numpy().astype(np.int32)
    del sim_tensor
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
    return original_dist, initial_rank


def compute_reranking_full_distance(original_dist: np.ndarray, initial_rank: np.ndarray, k1: int = 80, k2: int = 15, lambda_value: float = 0.2) -> np.ndarray:
    all_num = original_dist.shape[0]
    V = np.zeros((all_num, all_num), dtype=np.float16)
    half_k1 = int(np.round(k1 / 2.0))
    for index in range(all_num):
        forward = initial_rank[index, :k1 + 1]
        backward = initial_rank[forward, :k1 + 1]
        reciprocal = forward[np.any(backward == index, axis=1)]
        reciprocal_expansion = reciprocal.copy()
        for candidate in reciprocal:
            candidate_forward = initial_rank[candidate, :half_k1 + 1]
            candidate_backward = initial_rank[candidate_forward, :half_k1 + 1]
            candidate_reciprocal = candidate_forward[np.any(candidate_backward == candidate, axis=1)]
            if candidate_reciprocal.size == 0:
                continue
            overlap = np.intersect1d(candidate_reciprocal, reciprocal)
            if overlap.size > (2.0 / 3.0) * candidate_reciprocal.size:
                reciprocal_expansion = np.concatenate((reciprocal_expansion, candidate_reciprocal))
        reciprocal_expansion = np.unique(reciprocal_expansion)
        weights = np.exp(-original_dist[index, reciprocal_expansion]).astype(np.float32)
        V[index, reciprocal_expansion] = (weights / (weights.sum() + 1e-12)).astype(np.float16)
    if k2 > 1:
        V_qe = np.zeros_like(V, dtype=np.float16)
        for index in range(all_num):
            V_qe[index] = V[initial_rank[index, :k2]].mean(axis=0)
        V = V_qe
    inv_index = [np.flatnonzero(V[:, column]) for column in range(all_num)]
    jaccard_dist = np.zeros((all_num, all_num), dtype=np.float32)
    for index in range(all_num):
        temp_min = np.zeros(all_num, dtype=np.float32)
        non_zero = np.flatnonzero(V[index])
        for nz in non_zero:
            related = inv_index[nz]
            temp_min[related] += np.minimum(np.float32(V[index, nz]), V[related, nz].astype(np.float32))
        jaccard_dist[index] = 1.0 - temp_min / (2.0 - temp_min)
    return jaccard_dist * (1.0 - lambda_value) + original_dist * lambda_value


RERANK_CANONICAL = {"k1": 80, "k2": 15, "lambda_value": 0.2}
AQE_K_14T = 3
W_CLIPSENET_14T = 0.7
W_TRANSREID_14T = 0.3
S14T_PATH = FEATURE_DIR / "s14t_aqe3_rerank_similarity.npy"

transreid_matrix = l2_normalize(np.load(FEATURE_DIR / "transreid09v_cityflow_14u.npy").astype(np.float32))
clipsenet_matrix = l2_normalize(np.load(FEATURE_DIR / "clipsenet_cityflow_14u.npy").astype(np.float32))
combined_features = l2_normalize(np.concatenate([
    math.sqrt(W_CLIPSENET_14T) * clipsenet_matrix,
    math.sqrt(W_TRANSREID_14T) * transreid_matrix,
], axis=1).astype(np.float32))
aqe_features = average_query_expansion(combined_features, k=AQE_K_14T, iterations=1)
aqe_similarity = (aqe_features @ aqe_features.T).astype(np.float32)
state = build_rerank_state_from_similarity(aqe_similarity, max_k1=RERANK_CANONICAL["k1"])
rerank_dist = compute_reranking_full_distance(state[0], state[1], **RERANK_CANONICAL)
s14t_similarity = (1.0 - rerank_dist).astype(np.float32)
s14t_similarity = 0.5 * (s14t_similarity + s14t_similarity.T)
np.fill_diagonal(s14t_similarity, 1.0)
if s14t_similarity.shape != (EXPECTED_TRACKLETS, EXPECTED_TRACKLETS):
    raise RuntimeError(f"Unexpected S14t shape: {s14t_similarity.shape}")
if not np.isfinite(s14t_similarity).all():
    raise RuntimeError("S14t similarity contains NaN/Inf")
np.save(S14T_PATH, s14t_similarity)
write_json(FEATURE_DIR / "s14t_similarity_summary.json", {
    "path": str(S14T_PATH),
    "shape": list(s14t_similarity.shape),
    "w_clipsenet": W_CLIPSENET_14T,
    "w_transreid": W_TRANSREID_14T,
    "aqe": {"k": AQE_K_14T, "iterations": 1},
    "rerank": RERANK_CANONICAL,
    "min": float(s14t_similarity.min()),
    "max": float(s14t_similarity.max()),
    "mean": float(s14t_similarity.mean()),
})
print(json.dumps(json.loads((FEATURE_DIR / "s14t_similarity_summary.json").read_text()), indent=2))

## 8. Define Option C Sweep

In [ ]:
from src.core.config import load_config, save_config
from src.core.data_models import TrackletFeatures
from src.core.logging_utils import setup_logging
from src.stage3_indexing import run_stage3
from src.stage4_association import run_stage4
from src.stage5_evaluation import run_stage5

RUN_NAME = f"run_14u_cityflow_veri_fusion_v1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
setup_logging(level="INFO", log_file=RUN_DIR / "pipeline.log")
print(f"Run: {RUN_NAME}")

BASE_PRIMARY_WEIGHT = 0.475
BASE_TERTIARY_WEIGHT = 0.525
W_14T_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
SIMILARITY_THRESHOLDS = [0.46, 0.48, 0.50]
ANCHOR_CONFIG = {"aqe_k": 2, "fic_regularisation": 0.5}
SWEEP_CONFIGS = [{
    "config_id": "U0",
    "block": "drift",
    "w_14t": 0.0,
    "w_primary": BASE_PRIMARY_WEIGHT,
    "w_tertiary": BASE_TERTIARY_WEIGHT,
    "similarity_threshold": 0.48,
    "notes": "DRIFT GATE: reproduce 14e B1 / 14h M0 with S14t disabled by empty secondary path and zero weight",
    **ANCHOR_CONFIG,
}]
for w_14t in W_14T_VALUES:
    retained = 1.0 - w_14t
    for similarity_threshold in SIMILARITY_THRESHOLDS:
        SWEEP_CONFIGS.append({
            "config_id": f"U{len(SWEEP_CONFIGS)}",
            "block": "option_c_grid",
            "w_14t": w_14t,
            "w_primary": round(BASE_PRIMARY_WEIGHT * retained, 6),
            "w_tertiary": round(BASE_TERTIARY_WEIGHT * retained, 6),
            "similarity_threshold": similarity_threshold,
            "notes": f"Option C S14t stream weight {w_14t:.2f}; primary/tertiary ratio preserved from 14e B1",
            **ANCHOR_CONFIG,
        })
if len(SWEEP_CONFIGS) != 19:
    raise RuntimeError(f"Expected 19 configs (U0 + 18 grid), got {len(SWEEP_CONFIGS)}")
for config in SWEEP_CONFIGS:
    total_weight = float(config["w_primary"]) + float(config["w_tertiary"]) + float(config["w_14t"])
    if abs(total_weight - 1.0) > 1e-6:
        raise RuntimeError(f"Weights do not sum to 1.0 for {config['config_id']}: {total_weight}")

U0_REPRO_TARGET = 0.77936
U0_REPRO_TOL = 0.0005
U0_ID_SWITCH_TARGET = 154
WIN_THRESHOLD = 0.7810
MARGINAL_MIN = 0.7800
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70
APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)
BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38
INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30
MULTI_QUERY_WEIGHT = 0.00
MTMC_ONLY = False


def is_cityflow_gt_root(path: Path) -> bool:
    return path.exists() and all((path / cam / "gt" / "gt.txt").exists() for cam in EXPECTED_CAMS)


def find_cityflow_gt_root() -> Path:
    candidates = [
        PROJECT / "data" / "raw" / "cityflowv2",
        EXTRACT_DIR / "gt_annotations",
        Path("/kaggle/input/data-aicity-2023-track-2/train"),
        Path("/kaggle/input/data-aicity-2023-track-2"),
        Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2/train"),
        Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
    ]
    for candidate in candidates:
        if is_cityflow_gt_root(candidate):
            return candidate
    for gt_file in INPUT_ROOT.rglob("gt.txt") if INPUT_ROOT.exists() else []:
        if gt_file.parent.name != "gt" or gt_file.parent.parent.name not in EXPECTED_CAMS:
            continue
        candidate = gt_file.parents[2]
        if is_cityflow_gt_root(candidate):
            return candidate
    raise FileNotFoundError("CityFlowV2 GT root not found")


GT_DIR = find_cityflow_gt_root()
print(f"Ground truth root: {GT_DIR}")
print(json.dumps({"config_count": len(SWEEP_CONFIGS), "configs": SWEEP_CONFIGS}, indent=2))

## 9. Run Stage 3-5 Sweep

In [ ]:
def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "trackeval_idf1": payload.get("idf1"),
        "idp": payload.get("idp", details.get("idp")),
        "idr": payload.get("idr", details.get("idr")),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }


def summarize_prediction_dir(pred_dir: Path) -> dict:
    files = sorted(pred_dir.glob("*.txt")) if pred_dir.exists() else []
    rows_by_camera = {}
    ids_by_camera = {}
    for pred_file in files:
        rows = [line.strip().split(",") for line in pred_file.open() if line.strip()]
        rows_by_camera[pred_file.stem] = len(rows)
        ids_by_camera[pred_file.stem] = len({row[1] for row in rows if len(row) > 1})
    return {
        "exists": pred_dir.exists(),
        "file_count": len(files),
        "rows_by_camera": rows_by_camera,
        "ids_by_camera": ids_by_camera,
        "total_rows": int(sum(rows_by_camera.values())),
        "total_ids_camera_sum": int(sum(ids_by_camera.values())),
    }


def copy_recovery_artifacts(config_dir: Path, config_id: str) -> Path:
    recovery_dir = Path("/kaggle/working/outputs/14u_recovery") / config_id
    if recovery_dir.exists():
        shutil.rmtree(recovery_dir)
    recovery_dir.mkdir(parents=True, exist_ok=True)
    for rel_path in [
        Path("config.yaml"),
        Path("stage4/global_trajectories.json"),
        Path("stage4/forensic_report.json"),
        Path("stage5/evaluation_report.json"),
    ]:
        source_path = config_dir / rel_path
        if source_path.exists():
            target_path = recovery_dir / rel_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, target_path)
    pred_dir = config_dir / "stage5" / "predictions_mot"
    if pred_dir.exists():
        shutil.copytree(pred_dir, recovery_dir / "stage5" / "predictions_mot")
    return recovery_dir


def build_features(stage2_dir: Path) -> tuple[list[TrackletFeatures], dict]:
    index_map = json.loads((stage2_dir / "embedding_index.json").read_text(encoding="utf-8"))
    embeddings = np.load(stage2_dir / "embeddings.npy").astype(np.float32)
    hsv_features = np.load(stage2_dir / "hsv_features.npy").astype(np.float32)
    if embeddings.shape[0] != len(index_map) or hsv_features.shape[0] != len(index_map):
        raise ValueError(f"Stage2 row mismatch: embeddings={embeddings.shape}, hsv={hsv_features.shape}, index={len(index_map)}")
    features = [
        TrackletFeatures(
            track_id=int(row["track_id"]),
            camera_id=str(row["camera_id"]),
            class_id=int(row["class_id"]),
            embedding=embeddings[row_index],
            hsv_histogram=hsv_features[row_index],
            raw_embedding=None,
            multi_query_embeddings=None,
        )
        for row_index, row in enumerate(index_map)
    ]
    return features, {"num_features": len(features), "primary_shape": list(embeddings.shape), "hsv_shape": list(hsv_features.shape)}


def build_overrides(config: dict, config_run_name: str, stage2_dir: Path) -> list[str]:
    w_14t = float(config["w_14t"])
    w_tertiary = float(config["w_tertiary"])
    sim_thresh = float(config["similarity_threshold"])
    aqe_k = int(config["aqe_k"])
    fic_reg = float(config["fic_regularisation"])
    if w_14t == 0.0:
        secondary_path = ""
        secondary_weight = 0.0
    else:
        secondary_path = str(S14T_PATH)
        secondary_weight = w_14t
    return [
        f"project.run_name={config_run_name}",
        f"project.output_dir={DATA_OUT}",
        "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
        f"stage4.association.query_expansion.k={aqe_k}",
        "stage4.association.query_expansion.alpha=5.0",
        "stage4.association.query_expansion.dba=false",
        f"stage4.association.graph.similarity_threshold={sim_thresh}",
        f"stage4.association.solver={SOLVER}",
        f"stage4.association.graph.algorithm={ALGORITHM}",
        f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
        f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
        f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
        f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
        f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
        f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
        "stage4.association.mutual_nn.top_k_per_query=20",
        "stage4.association.fic.enabled=true",
        f"stage4.association.fic.regularisation={fic_reg}",
        "stage4.association.reranking.enabled=false",
        "stage4.association.camera_pair_norm.enabled=false",
        "stage4.association.fac.enabled=false",
        f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
        f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
        f"stage4.association.multi_query.dir={stage2_dir}",
        "stage4.association.aflink.enabled=false",
        f"stage4.association.secondary_embeddings.path={secondary_path}",
        f"stage4.association.secondary_embeddings.weight={secondary_weight}",
        f"stage4.association.tertiary_embeddings.path={stage2_dir / 'embeddings_tertiary.npy'}",
        f"stage4.association.tertiary_embeddings.weight={w_tertiary}",
        "stage4.association.camera_bias.enabled=false",
        "stage4.association.zone_model.enabled=false",
        "stage4.association.hierarchical.enabled=false",
        f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
        f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
        f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
        "stage4.association.gallery_expansion.enabled=true",
        f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
        f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
        "stage4.association.weights.length_weight_power=0.3",
        "stage4.association.temporal_overlap.enabled=true",
        "stage4.association.temporal_overlap.bonus=0.05",
        "stage4.association.temporal_overlap.max_mean_time=5.0",
        f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
        "stage5.stationary_filter.enabled=true",
        "stage5.stationary_filter.min_displacement_px=150",
        "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "stage5.min_submission_confidence=0.15",
        "stage5.cross_id_nms_iou=0.40",
        "stage5.min_trajectory_confidence=0.30",
        "stage5.min_trajectory_frames=40",
        "stage5.track_edge_trim.enabled=false",
        "stage5.track_smoothing.enabled=false",
        "stage5.gt_frame_clip=true",
        "stage5.gt_zone_filter=true",
        f"stage5.ground_truth_dir={GT_DIR}",
    ]


def run_config(config: dict) -> dict:
    config_id = config["config_id"]
    config_dir = RUN_DIR / config_id
    config_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 80)
    print(f"Running {config_id}: w_primary={config['w_primary']:.5f}, w_tertiary={config['w_tertiary']:.5f}, w_14t={config['w_14t']:.5f}, sim_thresh={config['similarity_threshold']:.2f}")
    print("=" * 80)
    tracklets = load_tracklets_by_camera(SOURCE_STAGE1_DIR)
    features, feature_summary = build_features(SOURCE_STAGE2_DIR)
    cfg = load_config(
        "configs/default.yaml",
        dataset_config="configs/datasets/cityflowv2.yaml",
        overrides=build_overrides(config, f"{RUN_NAME}_{config_id}", SOURCE_STAGE2_DIR),
    )
    save_config(cfg, config_dir / "config.yaml")
    start = time.time()
    faiss_index, metadata_store = run_stage3(cfg, features, tracklets, output_dir=config_dir / "stage3")
    stage3_min = (time.time() - start) / 60.0
    start = time.time()
    trajectories = run_stage4(cfg, faiss_index, metadata_store, features, tracklets, output_dir=config_dir / "stage4")
    stage4_min = (time.time() - start) / 60.0
    start = time.time()
    run_stage5(cfg, trajectories, output_dir=config_dir / "stage5")
    stage5_min = (time.time() - start) / 60.0
    report_path = config_dir / "stage5" / "evaluation_report.json"
    metrics = load_metrics(report_path)
    prediction_summary = summarize_prediction_dir(config_dir / "stage5" / "predictions_mot")
    recovery_dir = copy_recovery_artifacts(config_dir, config_id)
    idf1_value = metrics.get("mtmc_idf1") if metrics.get("mtmc_idf1") is not None else metrics.get("trackeval_idf1")
    if idf1_value is None:
        raise RuntimeError(f"IDF1 not found in {report_path}")
    row = {
        "config_id": config_id,
        "block": config["block"],
        "w_primary": float(config["w_primary"]),
        "w_tertiary": float(config["w_tertiary"]),
        "w_14t": float(config["w_14t"]),
        "similarity_threshold": float(config["similarity_threshold"]),
        "aqe_k": int(config["aqe_k"]),
        "fic_regularisation": float(config["fic_regularisation"]),
        "notes": config["notes"],
        "mtmc_idf1": metrics.get("mtmc_idf1"),
        "trackeval_idf1": metrics.get("trackeval_idf1"),
        "idp": metrics.get("idp"),
        "idr": metrics.get("idr"),
        "id_switches": metrics.get("id_switches"),
        "fragmentations": metrics.get("fragmentations"),
        "mota": metrics.get("mota"),
        "hota": metrics.get("hota"),
        "conflations": metrics.get("conflations"),
        "num_pred_ids": metrics.get("num_pred_ids"),
        "num_trajectories": len(trajectories),
        "num_stage4_tracklets": int(sum(len(trajectory.tracklets) for trajectory in trajectories)),
        "prediction_summary": prediction_summary,
        "feature_summary": feature_summary,
        "stage_timings_min": {"stage3": round(stage3_min, 2), "stage4": round(stage4_min, 2), "stage5": round(stage5_min, 2)},
        "paths": {"config_dir": str(config_dir), "evaluation_report": str(report_path), "recovery_dir": str(recovery_dir)},
    }
    print(f"{config_id} MTMC IDF1: {float(idf1_value):.5f} id_switches={row['id_switches']}")
    return row


results = []
sweep_results = []
halt_reason = None
drift_detected = False
wall_start = time.time()

drift_check_result = run_config(SWEEP_CONFIGS[0])
results.append(drift_check_result)
(RUN_DIR / "14u_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

u0_idf1 = float(drift_check_result["mtmc_idf1"])
u0_id_switches = drift_check_result.get("id_switches")
if abs(u0_idf1 - U0_REPRO_TARGET) > U0_REPRO_TOL or u0_id_switches != U0_ID_SWITCH_TARGET:
    drift_detected = True
    halt_reason = f"U0 drift gate failed: got idf1={u0_idf1:.5f}, id_switches={u0_id_switches}; expected {U0_REPRO_TARGET:.5f} +/- {U0_REPRO_TOL:.5f} and id_switches={U0_ID_SWITCH_TARGET}"
    print(halt_reason)
else:
    print(f"U0 drift gate passed: {u0_idf1:.5f}, id_switches={u0_id_switches}")
    for config in SWEEP_CONFIGS[1:]:
        row = run_config(config)
        results.append(row)
        sweep_results.append(row)
        (RUN_DIR / "14u_partial_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

overall_best = max(results, key=lambda row: row["mtmc_idf1"] if row["mtmc_idf1"] is not None else -1.0)
best_14t = max(sweep_results, key=lambda row: row["mtmc_idf1"] if row["mtmc_idf1"] is not None else -1.0) if sweep_results else None
if drift_detected:
    verdict = "DRIFT"
elif float(overall_best["mtmc_idf1"]) >= WIN_THRESHOLD:
    verdict = "WIN"
elif float(overall_best["mtmc_idf1"]) >= MARGINAL_MIN:
    verdict = "MARGINAL"
else:
    verdict = "FAIL"

print("\n" + "#" * 80)
print(f"BEST 14u CONFIG: {overall_best['config_id']} w_14t={overall_best['w_14t']:.3f} thr={overall_best['similarity_threshold']:.2f} MTMC_IDF1={overall_best['mtmc_idf1']:.5f} id_switches={overall_best.get('id_switches')} verdict={verdict} drift={drift_detected}")
if halt_reason:
    print(f"HALT: {halt_reason}")
print("#" * 80)

## 10. Persist Summary

In [ ]:
summary_dir = Path("/kaggle/working/outputs/14u_v1_summary")
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = Path("/kaggle/working/14u_summary.json")
summary_output_path = summary_dir / "14u_summary.json"
eval_results_path = Path("/kaggle/working/eval_results.json")
recipe_path = Path("/kaggle/working/recipe.json")

compact_configs = [
    {
        "config_id": row.get("config_id"),
        "w_primary": row.get("w_primary"),
        "w_tertiary": row.get("w_tertiary"),
        "w_14t": row.get("w_14t"),
        "similarity_threshold": row.get("similarity_threshold"),
        "mtmc_idf1": row.get("mtmc_idf1"),
        "trackeval_idf1": row.get("trackeval_idf1"),
        "id_switches": row.get("id_switches"),
        "mota": row.get("mota"),
        "hota": row.get("hota"),
    }
    for row in results
]

summary = {
    "experiment": "14u-cityflow-veri-fusion-port-option-c",
    "kernel": "yahiaakhalafallah/14u-cityflow-veri-fusion-port",
    "run_name": RUN_NAME,
    "source_14h_run_name": SOURCE_14H_RUN_NAME,
    "frame_id_convention": "0-based internal Stage 1/2 frame IDs; MOT output is converted to 1-based in Stage 5",
    "option": "C",
    "verdict": verdict,
    "drift_detected": drift_detected,
    "drift_check_config_id": "U0",
    "drift_check_result": drift_check_result,
    "drift_reproduction_target": U0_REPRO_TARGET,
    "drift_reproduction_tolerance": U0_REPRO_TOL,
    "drift_id_switch_target": U0_ID_SWITCH_TARGET,
    "configs": compact_configs,
    "planned_configs": SWEEP_CONFIGS,
    "results": results,
    "sweep_results": sweep_results,
    "overall_best": overall_best,
    "best": overall_best,
    "best_14t_stream": best_14t,
    "halt_reason": halt_reason,
    "total_config_count": len(SWEEP_CONFIGS),
    "executed_config_count": len(results),
    "feature_sources": {
        "primary_and_tertiary_kernel": SOURCE_14H_OWNER_SLUG,
        "clipsenet_kernel": SOURCE_13_OWNER_SLUG,
        "transreid_dataset": "mrkdagods/mtmc-weights",
        "transreid_checkpoint": str(TRANSREID_WEIGHTS),
        "clipsenet_checkpoint": str(CLIPSENET_WEIGHTS),
        "s14t_similarity_path": str(S14T_PATH),
        "stage1_tracklets": str(SOURCE_STAGE1_DIR),
        "stage2_anchor_features": str(SOURCE_STAGE2_DIR),
        "dual_feature_dir": str(FEATURE_DIR),
    },
    "fixed_config": {
        "pca_components": 384,
        "algorithm": ALGORITHM,
        "aqe_k": 2,
        "fic_regularisation": 0.5,
        "gallery_expansion": True,
        "gallery_threshold": GALLERY_THRESH,
        "orphan_match_threshold": ORPHAN_MATCH_THRESH,
        "temporal_overlap_bonus": 0.05,
        "intra_merge": INTRA_MERGE,
        "intra_merge_threshold": INTRA_MERGE_THRESH,
        "intra_merge_gap": INTRA_MERGE_GAP,
        "mtmc_only_submission": MTMC_ONLY,
        "score_fusion_math": "appearance_sim = w_p*S_primary + w_t*S_DINOv2 + w_14t*S14t_precomputed; w_p/w_t preserve 14e B1 ratio under retained mass (1-w_14t)",
        "s14t_internal_recipe": {"w_clipsenet": W_CLIPSENET_14T, "w_transreid": W_TRANSREID_14T, "aqe_k": AQE_K_14T, "rerank": RERANK_CANONICAL},
    },
    "sweep_grid": {"w_14t": W_14T_VALUES, "similarity_threshold": SIMILARITY_THRESHOLDS, "grid_size_excluding_u0": len(SWEEP_CONFIGS) - 1},
    "verdict_bands": {"WIN": f">={WIN_THRESHOLD}", "MARGINAL": f"[{MARGINAL_MIN},{WIN_THRESHOLD})", "FAIL": f"<{MARGINAL_MIN}", "DRIFT": "U0 outside tolerance or id_switches mismatch"},
    "stage_timings_min": {"stage345_sweep": round((time.time() - wall_start) / 60.0, 2)},
    "paths": {"run_dir": str(RUN_DIR), "summary": str(summary_path), "summary_output": str(summary_output_path), "eval_results": str(eval_results_path), "recipe": str(recipe_path)},
}

recipe = {
    "experiment": summary["experiment"],
    "option": "C",
    "models": MODEL_INFO,
    "cityflow_dataset": "thanhnguyenle/data-aicity-2023-track-2",
    "source_14h": SOURCE_14H_OWNER_SLUG,
    "source_clipsenet": SOURCE_13_OWNER_SLUG,
    "feature_extraction": {"image_size": [224, 224], "tta_views": ["original", "hflip", "scale_0.95", "scale_1.05"], "pooling": "softmax_quality_mean_temperature_3", "samples_per_tracklet": SAMPLES_PER_TRACKLET},
    "s14t_similarity": {"formula": "rerank(AQE(L2([sqrt(0.7)*CLIP-SENet, sqrt(0.3)*TransReID09v]), k=3), k1=80, k2=15, lambda=0.2)", "path": str(S14T_PATH)},
    "sweep_configs": SWEEP_CONFIGS,
    "fixed_stage4": summary["fixed_config"],
}

summary_payload = json.dumps(summary, indent=2)
summary_path.write_text(summary_payload, encoding="utf-8")
summary_output_path.write_text(summary_payload, encoding="utf-8")
eval_results_path.write_text(json.dumps({"rows": results, "planned_configs": SWEEP_CONFIGS}, indent=2), encoding="utf-8")
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(f"Saved summary: {summary_path}")
print(f"Saved eval results: {eval_results_path}")
print(f"Saved recipe: {recipe_path}")
print(json.dumps(summary, indent=2))